In [1]:
# python 3.12.7
# import bibliotek

import os
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, f1_score,
    recall_score, precision_score, confusion_matrix, roc_curve, auc, precision_recall_curve, 
)
from pathlib import Path
from sklearn.preprocessing import StandardScaler  
from sklearn.impute import SimpleImputer          
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV


### **XGBOOST Z OPTYMALIZACJĄ PROGU - DLA company_years_h2.parquet**


In [2]:
# --- 1. Wczytaj dane ---
# Znajdź plik w projekcie i pobierz nazwę jego katalogu
pliki = list(Path.cwd().rglob('company_years_h2.parquet'))
KATALOG_DANYCH = pliki[0].parent if pliki else "V4FinBench_data_load-main"

# Wczytaj plik company_years_h2.parquet
print(f"\n📁 Wczytywanie danych z company_years_h2.parquet...")
df_h2 = pd.read_parquet(os.path.join(KATALOG_DANYCH, 'company_years_h2.parquet'))

print(f"✅ Dane wczytane:")
print(f"  Shape: {df_h2.shape}")
print(f"  Kolumny: {df_h2.columns.tolist()[:10]}...")
print(f"  main_label: {df_h2['main_label'].sum():,} przypadków ({(df_h2['main_label'].mean()*100):.2f}%)")



📁 Wczytywanie danych z company_years_h2.parquet...


FileNotFoundError: [Errno 2] No such file or directory: 'V4FinBench_data_load-main\\company_years_h2.parquet'

In [ ]:
# --- 2. Podział czasowy: train / walidacja / test ---
# Zbiór TEST (lata > SPLIT_YEAR) służy WYŁĄCZNIE do końcowej oceny.
# Wszystkie decyzje (liczba cech, wybór modelu, próg) podejmujemy na WALIDACJI = ostatni rok treningowy.
SPLIT_YEAR = 2016
VAL_YEAR = SPLIT_YEAR
print(f"\n📅 Podział danych: train < {VAL_YEAR}, walidacja = {VAL_YEAR}, test > {SPLIT_YEAR}")

train = df_h2[df_h2['year'] < VAL_YEAR]
val = df_h2[df_h2['year'] == VAL_YEAR]
test = df_h2[df_h2['year'] > SPLIT_YEAR]

print(f"\n  Train: {train.shape} — positive rate: {train['main_label'].mean():.4%}")
print(f"  Val:   {val.shape} — positive rate: {val['main_label'].mean():.4%} ({val['main_label'].sum():,} bankrutów)")
print(f"  Test:  {test.shape} — positive rate: {test['main_label'].mean():.4%}")


In [ ]:
# --- 3. Przygotowanie cech ---
# Jawnie odrzucamy zmienne celu, identyfikatory oraz metadane
DROP_COLS = [
    'main_label',
    'Insolvency_flag',
    'num',
    'emis_id',
    'year',
    'company',
    'industry',
    'link',
    'operational_status'
]

feature_cols = [
    c for c in df_h2.select_dtypes(include='number').columns
    if c not in DROP_COLS
]

print(f"\n  Liczba cech: {len(feature_cols)}")

X_train, y_train = train[feature_cols], train['main_label']
X_val, y_val = val[feature_cols], val['main_label']
X_test, y_test = test[feature_cols], test['main_label']

print(f"\n  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")


In [ ]:
# --- 4. Sprawdzenie braków danych ---
for nazwa, X in [('train', X_train), ('val', X_val), ('test', X_test)]:
    braki = X.isnull().sum().sum()
    print(f"  Braki w {nazwa}: {braki:,} ({braki/(X.shape[0]*X.shape[1])*100:.2f}%)")


In [ ]:
# --- 5. Wykonywanie hybrydowego downsamplingu ---
print("\n🔄 Wykonywanie hybrydowego downsamplingu na zbiorze TRAIN...")

# Połączenie X_train i y_train
train_concat = pd.concat([X_train, y_train], axis=1)
majority = train_concat[train_concat['main_label'] == 0]
minority = train_concat[train_concat['main_label'] == 1]

# Stosunek 1:5 (1 bankrut na 5 zdrowych firm)
RATIO_TARGET = 5  
n_target_majority = len(minority) * RATIO_TARGET

majority_downsampled = resample(
    majority, 
    replace=False, 
    n_samples=min(len(majority), n_target_majority), 
    random_state=42
)

# Nowy zbalansowany zbiór treningowy
train_balanced = pd.concat([majority_downsampled, minority])
X_train_res = train_balanced.drop('main_label', axis=1)
y_train_res = train_balanced['main_label']

print(f"✅ Stary Train: {X_train.shape[0]:,} wierszy (Positive rate: {y_train.mean():.4%})")
print(f"✅ Nowy Train:  {X_train_res.shape[0]:,} wierszy (Positive rate: {y_train_res.mean():.4%})")


### **5b. Selekcja cech (Feature Selection)**


In [ ]:
# --- 5b. Automatyczna selekcja cech (SelectFromModel) ---
print("\n🔍 Przeprowadzanie automatycznej selekcji cech (SelectFromModel + XGBoost)...")

# 1. Trenujemy szybki model do oceny cech na zbalansowanym zbiorze
fs_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss')
fs_model.fit(X_train_res, y_train_res)

# 2. SelectFromModel odrzuca cechy słabe (poniżej MEDIANY ważności -> zostaje ok. połowa cech)
selector = SelectFromModel(fs_model, prefit=True, threshold='median')

# 3. Odczytujemy, co wybrał model
selected_mask = selector.get_support()
top_features = X_train_res.columns[selected_mask].tolist()

print(f"✅ Model automatycznie zachował {len(top_features)} cech (ważność >= mediany) z {X_train_res.shape[1]}.")

# Wykres podsumowujący
feature_importances = pd.Series(fs_model.feature_importances_, index=X_train_res.columns)
plt.figure(figsize=(12, 5))
feature_importances[top_features].sort_values(ascending=False).head(20).plot(kind='bar', color='skyblue')
plt.title(f"Top 20 najważniejszych z {len(top_features)} automatycznie wybranych cech")
plt.ylabel("Ważność (Feature Importance)")
plt.show()

# 4. Zastąpienie starych zbiorów przefiltrowanymi
X_train_res_full = X_train_res.copy() # Kopie zapasowe pełnych zbiorów
X_val_full = X_val.copy()
X_test_full = X_test.copy()

# Aktualizujemy zbiory o wybrane (automatycznie) cechy
X_train_res = X_train_res[top_features]
X_val = X_val[top_features]
X_test = X_test[top_features]
X_train = X_train[top_features]

print("✅ Zbiory danych pomyślnie zaktualizowane.")


In [ ]:
# --- 5c. Ręczna selekcja cech (Testowanie liczby optymalnych cech na WALIDACJI) ---
print("1. Trenowanie bazowego modelu do oceny ważności wszystkich cech...")
xgb_feat = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_feat.fit(X_train_res, y_train_res)

# Zbieramy ważność wszystkich cech
importances = pd.Series(xgb_feat.feature_importances_, index=X_train_res.columns)
top_features = importances.sort_values(ascending=False).index.tolist()

print("2. Testowanie różnej liczby TOP cech na zbiorze WALIDACYJNYM (test pozostaje nietknięty)...")
results_feat = []
ks = [k for k in [10, 15, 20, 30, 50] if k < len(top_features)] + [len(top_features)]

for k in ks:
    features_k = top_features[:k]

    # Trenujemy mały model tylko na tych 'k' cechach
    model_k = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    model_k.fit(X_train_res[features_k], y_train_res)

    # Oceniamy na walidacji
    proba_k = model_k.predict_proba(X_val[features_k])[:, 1]

    results_feat.append({
        'Liczba cech': k if k != len(top_features) else 'Wszystkie',
        'AUC (walidacja)': round(roc_auc_score(y_val, proba_k), 4),
        'Avg Precision (walidacja)': round(average_precision_score(y_val, proba_k), 4)
    })

df_feat_results = pd.DataFrame(results_feat)
display(df_feat_results)

# --- WYBÓR OSTATECZNY ---
FINAL_K = 15  # <--- TUTAJ MOŻESZ ZMIENIĆ, ILE CECH CHCESZ ZOSTAWIĆ PO ZOBACZENIU TABELKI (walidacja!)
final_features = top_features[:FINAL_K]

X_train_res = X_train_res[final_features]
X_val = X_val[final_features]
X_test = X_test[final_features]
X_train = X_train[final_features]

print(f"\n✅ Zbiór danych został pomyślnie przycięty do najlepszych {FINAL_K} cech. Możesz przejść do kolejnych kroków!")


In [ ]:
# --- 6. Porównanie Algorytmów (Benchmark) — na zbiorze WALIDACYJNYM ---
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost (Baseline)': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

print("Imputacja braków danych dla modeli innych niż XGBoost...")
imputer = SimpleImputer(strategy='median')
X_train_res_imp = pd.DataFrame(imputer.fit_transform(X_train_res), columns=X_train_res.columns)
X_val_imp = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)

print("\n📊 Trenowanie modeli do tabeli porównawczej (ocena na walidacji)...")

results = []

for name, m in models.items():
    if 'XGBoost' in name:
        m.fit(X_train_res, y_train_res)
        proba_val = m.predict_proba(X_val)[:, 1]
    else:
        m.fit(X_train_res_imp, y_train_res)
        proba_val = m.predict_proba(X_val_imp)[:, 1]

    # --- OPTYMALIZACJA PROGU (na walidacji) ---
    precisions, recalls, thresholds = precision_recall_curve(y_val, proba_val)

    # F1 dla każdego progu (ostatni punkt krzywej nie ma progu, więc go pomijamy)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1])
    f1_scores = np.nan_to_num(f1_scores) # obsługa dzielenia przez zero

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    y_pred_val = (proba_val >= best_threshold).astype(int)

    results.append({
        'Model': name,
        'Opt. Threshold': round(best_threshold, 4),
        'F1-Score': round(f1_score(y_val, y_pred_val), 4),
        'Recall': round(recall_score(y_val, y_pred_val), 4),
        'Precision': round(precision_score(y_val, y_pred_val, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_val, proba_val), 4),
        'Avg Precision': round(average_precision_score(y_val, proba_val), 4)
    })

display(pd.DataFrame(results))


In [ ]:
# --- 8.5 OPTYMALIZACJA HIPERPARAMETRÓW (RandomizedSearchCV) ---


print("Rozpoczynam szukanie najlepszych parametrów dla XGBoost...")

# 1. Definiujemy, jakie parametry chcemy przetestować
param_grid = {
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}


# 2. Tworzymy "pusty" model
xgb_tuner = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# 3. Odpalamy losowe szukanie (np. 10 najciekawszych kombinacji), żeby nie czekać do jutra
random_search = RandomizedSearchCV(
    estimator=xgb_tuner, 
    param_distributions=param_grid, 
    n_iter=10,            # Sprawdzi 10 losowych kombinacji
    scoring='roc_auc',    # Szukamy pod maksymalne AUC
    cv=3,                 # 3-krotna walidacja krzyżowa
    verbose=2,
    random_state=42,
    n_jobs=-1             # Używa wszystkich rdzeni procesora
)

# 4. Trenujemy! (Może to potrwać parę minut)
random_search.fit(X_train_res, y_train_res)

print("\n🎉 ZNALAZŁEM NAJLEPSZE PARAMETRY:")
print(random_search.best_params_)


# --- 8.5 OPTYMALIZACJA HIPERPARAMETRÓW (RandomizedSearchCV) --- Glebsze szukanie ale trwa bardzo  dlugo 
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

print("Rozpoczynam rozszerzone szukanie najlepszych parametrów dla XGBoost...")

# 1. Definiujemy rozszerzoną siatkę (więcej możliwości)
param_grid = {
    'max_depth': [3, 4, 6, 8, 10, 12],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.5, 0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

# 2. Tworzymy "pusty" model
xgb_tuner = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# 3. Odpalamy losowe szukanie - uwaga, n_iter zmienione na 30!
random_search = RandomizedSearchCV(
    estimator=xgb_tuner, 
    param_distributions=param_grid, 
    n_iter=30,            # Zwiększone do 30 losowych kombinacji (dłuższy czas, lepsze wyniki)
    scoring='roc_auc',    # Szukamy pod maksymalne AUC
    cv=3,                 # 3-krotna walidacja krzyżowa
    verbose=2,
    random_state=42,
    n_jobs=-1             # Używa wszystkich rdzeni procesora
)

# 4. Trenujemy! (Może to potrwać np. 10-15 minut)
random_search.fit(X_train_res, y_train_res)

print("\n🎉 ZNALAZŁEM NAJLEPSZE PARAMETRY:")
print(random_search.best_params_)


In [ ]:
# --- 7. TRENOWANIE OSTATECZNEGO ZOPTYMALIZOWANEGO MODELU ---
print("Trenowanie ostatecznego modelu z najlepszymi hiperparametrami...")
# Pobieramy najlepsze parametry znalezione przed chwilą i wrzucamy do modelu
model = xgb.XGBClassifier(**random_search.best_params_, random_state=42, eval_metric='logloss')

# Trenujemy na zbalansowanym zbiorze
model.fit(X_train_res, y_train_res)

# Prawdopodobieństwa: walidacja (do wyboru progu) i test (tylko do oceny)
y_proba_val = model.predict_proba(X_val)[:, 1]
y_proba_test = model.predict_proba(X_test)[:, 1]
y_proba = y_proba_test # alias używany w krokach 9-13

print("✅ Gotowe! Ostateczny model przetrenowany.")


In [ ]:
# --- 8. Metryki dla domyślnego progu (ostateczny model, zbiór testowy) ---
print("\n" + "="*70)
print("📈 METRYKI (próg domyślny = 0.5)")
print("="*70)

y_pred_default = (y_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_proba)
avg_prec = average_precision_score(y_test, y_proba)
f1_default = f1_score(y_test, y_pred_default)
recall_default = recall_score(y_test, y_pred_default)
precision_default = precision_score(y_test, y_pred_default, zero_division=0)

print(f"\n  ROC-AUC:        {roc_auc:.4f}")
print(f"  Avg Precision:  {avg_prec:.4f}")
print(f"  F1 Score:       {f1_default:.4f}")
print(f"  Recall:         {recall_default:.4f}")
print(f"  Precision:      {precision_default:.4f}")


In [ ]:
# --- 9. OPTYMALIZACJA PROGU (NA WALIDACJI) I OSTATECZNA EWALUACJA (TEST) ---
print("======================================================================")
print("🎯 PODEJŚCIE BIZNESOWE (BANKOWE) - CELUJEMY W WYŁAPANIE 80% BANKRUTÓW")
print("======================================================================")

# Próg wyznaczamy na walidacji: model nie widział tych firm-lat podczas treningu,
# więc recall jest realistyczny (na danych treningowych byłby zawyżony).
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba_val)

# 🎯 Podejście Biznesowe: Szukamy progu, przy którym Recall wynosi minimum 80%
target_recall = 0.80
valid_indexes = np.where(recalls[:-1] >= target_recall)[0]

if len(valid_indexes) > 0:
    # Ostatni indeks z recall >= 80% = najwyższy taki próg (najwyższa możliwa precyzja)
    best_idx = valid_indexes[-1]
    best_thresh = thresholds[best_idx]
else:
    best_thresh = thresholds[0]

print(f"🎯 Biznesowy próg (aby wyłapać min. {target_recall:.0%} bankrutów na WALIDACJI): {best_thresh:.4f}")

# Aplikacja progu do zbioru testowego
y_pred_optimal = (y_proba_test >= best_thresh).astype(int)

print("\n📊 WYNIKI BIZNESOWE NA ZBIORZE TESTOWYM:")
print(f"  * Recall:    {recall_score(y_test, y_pred_optimal):.4f} (Ilu bankrutów udało się złapać?)")
print(f"  * Precision: {precision_score(y_test, y_pred_optimal, zero_division=0):.4f} (Jaka część oflagowanych firm to faktycznie bankruci?)")
print(f"  * F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}")
print(f"  * ROC-AUC:   {roc_auc_score(y_test, y_proba_test):.4f}")

print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred_optimal))

cm = confusion_matrix(y_test, y_pred_optimal)
print("Macierz pomyłek:")
print(cm)


In [ ]:
# --- 10. Wyniki dla różnych progów ---
print("\n" + "="*70)
print("📊 WYNIKI DLA RÓŻNYCH PROGÓW:")
print("="*70)

for thresh in [0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_temp = (y_proba >= thresh).astype(int)
    rec = recall_score(y_test, y_pred_temp)
    prec = precision_score(y_test, y_pred_temp, zero_division=0)
    f1 = f1_score(y_test, y_pred_temp)
    print(f"  Próg {thresh:.2f}: recall={rec:.4f}, precision={prec:.4f}, f1={f1:.4f}")


In [ ]:
# --- 11. Macierz błędów dla najlepszego progu ---
print("\n" + "="*70)
print(f"📊 MACIERZ BŁĘDÓW (próg = {best_thresh:.2f})")
print("="*70)

y_pred_best = (y_proba >= best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_best)

print("\n  " + "-"*40)
print("  |              | Przewidziane |")
print("  |              |  0    |  1   |")
print("  |" + "-"*40)
print(f"  | Rzeczywiste 0 | {cm[0,0]:>4} | {cm[0,1]:>4} |")
print(f"  | Rzeczywiste 1 | {cm[1,0]:>4} | {cm[1,1]:>4} |")
print("  " + "-"*40)

tn, fp, fn, tp = cm.ravel()
print(f"\n  Interpretacja:")
print(f"    True Positives (TP):  {tp:,}  - poprawnie wykryte")
print(f"    False Positives (FP): {fp:,}  - fałszywe alarmy")
print(f"    False Negatives (FN): {fn:,}  - pominięte")
print(f"    True Negatives (TN):  {tn:,}  - poprawnie odrzucone")


In [ ]:
# --- 12. Wizualizacja Wyników ---
print("\n📊 Tworzenie wykresów...")

# Generujemy dane do wykresu 11b na żywo, aby uniknąć błędów z brakującą zmienną results
thresh_range = np.arange(0.01, 0.95, 0.01)
eval_results = []

for thresh in thresh_range:
    pred_t = (y_proba >= thresh).astype(int)
    eval_results.append({
        'threshold': thresh,
        'recall': recall_score(y_test, pred_t),
        'precision': precision_score(y_test, pred_t, zero_division=0),
        'f1': f1_score(y_test, pred_t)
    })

df_results = pd.DataFrame(eval_results)

# Tworzenie siatki wykresów 2x2
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 11a. Krzywa ROC
ax1 = axes[0, 0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax1.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}', linewidth=2)
ax1.plot([0, 1], [0, 1], 'k--', label='Losowy')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Krzywa ROC')
ax1.legend()
ax1.grid(True)

# 11b. Metryki vs próg decyzyjny
ax2 = axes[0, 1]
ax2.plot(df_results['threshold'], df_results['recall'], 'b-', label='Recall', linewidth=2)
ax2.plot(df_results['threshold'], df_results['precision'], 'r-', label='Precision', linewidth=2)
ax2.plot(df_results['threshold'], df_results['f1'], 'g-', label='F1', linewidth=2)
ax2.axvline(x=best_thresh, color='black', linestyle='--', label=f'Optymalny próg = {best_thresh:.2f}')
ax2.set_xlabel('Próg decyzyjny')
ax2.set_ylabel('Score')
ax2.set_title('Metryki dla różnych progów')
ax2.legend()
ax2.grid(True)

# 11c. Rozkład prawdopodobieństw
ax3 = axes[1, 0]
ax3.hist(y_proba[y_test == 0], bins=50, alpha=0.5, label='Klasa 0 (Zdrowe)', color='blue', density=True)
ax3.hist(y_proba[y_test == 1], bins=50, alpha=0.5, label='Klasa 1 (Bankrut)', color='red', density=True)
ax3.axvline(x=best_thresh, color='black', linestyle='--', label=f'Optymalny próg = {best_thresh:.2f}')
ax3.set_xlabel('Prawdopodobieństwo klasy 1')
ax3.set_ylabel('Gęstość')
ax3.set_title('Rozkład prawdopodobieństw')
ax3.legend()
ax3.grid(True)

# 11d. Macierz błędów (heatmap)
ax4 = axes[1, 1]
cm_best = confusion_matrix(y_test, (y_proba >= best_thresh).astype(int))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Klasa 0', 'Klasa 1'],
            yticklabels=['Klasa 0', 'Klasa 1'],
            ax=ax4)
ax4.set_title(f'Macierz błędów (próg = {best_thresh:.2f})')
ax4.set_xlabel('Przewidziane')
ax4.set_ylabel('Rzeczywiste')

plt.tight_layout()
plt.show()


### **Analiza wyniku dla modelu h2 (2 lata do przodu)**

Ostateczna ewaluacja modelu XGBoost na nienaruszonym zbiorze testowym przy progu decyzyjnym wyznaczonym na zbiorze **walidacyjnym**.
Liczby poniżej są generowane automatycznie z wyników powyżej, więc zawsze zgadzają się z bieżącym uruchomieniem.


In [ ]:
# --- 13. Podsumowanie biznesowe (liczone automatycznie) ---
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_optimal).ravel()
rec = tp / (tp + fn)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0

podsumowanie = pd.DataFrame({
    'Wartość': [
        f"{roc_auc_score(y_test, y_proba_test):.4f}",
        f"{average_precision_score(y_test, y_proba_test):.4f}",
        f"{rec:.2%} ({tp:,} / {tp + fn:,})",
        f"{prec:.2%} ({tp:,} / {tp + fp:,})",
        f"{f1_score(y_test, y_pred_optimal):.4f}",
        f"{fp:,}",
        f"{fn:,}",
        f"{tn / (tn + fp):.2%}",
    ]
}, index=['ROC-AUC', 'Avg Precision', 'Recall', 'Precision', 'F1-Score',
          'Fałszywe alarmy (FP)', 'Pominięte (FN)', 'Specyficzność (zdrowe przepuszczone)'])

print(f"Próg decyzyjny (z walidacji): {best_thresh:.4f}\n")
display(podsumowanie)

print(f"\nScenariusz: 100 firm, które zbankrutują za 2 lata")
print(f"  ✅ Model wykryje: ~{rec * 100:.0f} firm")
print(f"  ❌ Model pominie: ~{(1 - rec) * 100:.0f} firm")
print(f"\nNa każde 1,000 ostrzeżeń wygenerowanych przez model:")
print(f"  ✅ Prawdziwe bankructwa: ~{prec * 1000:.0f}")
print(f"  ⚠️ Fałszywe alarmy (do weryfikacji analityka): ~{(1 - prec) * 1000:.0f}")
